In [5]:
# =======================
# IEOR4004 Project – Q1 (Final v3)
# Facility-level model: piecewise expansion (0–100%, 100–120%) → up to 2.2×n_f
# New & expansion 0–5 slots both pay $100 equipment fee
# =======================

import pandas as pd
import numpy as np
from gurobipy import Model, GRB, quicksum

# ---------- 1. Load datasets ----------
base = "/Users/cuilinnan/Desktop/ChildCareDeserts_Data/"
df_fac = pd.read_csv(base + "child_care_regulated.csv")
df_inc = pd.read_csv(base + "avg_individual_income.csv")
df_pop = pd.read_csv(base + "population.csv")
df_emp = pd.read_csv(base + "employment_rate.csv")
df_loc = pd.read_csv(base + "potential_locations.csv")

# ---------- 2. Standardize ZIP codes ----------
df_fac.loc[df_fac["zip_code"] >= 100000, "zip_code"] = df_fac["zip_code"] // 10000
df_inc.loc[df_inc["ZIP code"] >= 100000, "ZIP code"] = df_inc["ZIP code"] // 10000
df_pop.loc[df_pop["zipcode"] >= 100000, "zipcode"] = df_pop["zipcode"] // 10000
df_emp.loc[df_emp["zipcode"] >= 100000, "zipcode"] = df_emp["zipcode"] // 10000
df_loc.loc[df_loc["zipcode"] >= 100000, "zipcode"] = df_loc["zipcode"] // 10000

# ---------- 3. Compute existing capacities ----------
df_fac["existing_capacity_0_12"] = df_fac["total_capacity"]

df_fac["existing_capacity_0_5"] = (
    df_fac[["infant_capacity", "toddler_capacity", "preschool_capacity"]].sum(axis=1)
    + (5/12) * df_fac["children_capacity"]
)

cap_by_zip = (
    df_fac.groupby("zip_code")[["existing_capacity_0_12","existing_capacity_0_5"]]
    .sum().reset_index()
)

# ---------- 4. Population adjustment ----------
df_pop["pop_0_5"] = df_pop["-5"]
df_pop["10-12"]   = df_pop["10-14"] * 3/5
df_pop["pop_0_12"] = df_pop[["-5","5-9","10-12"]].sum(axis=1)
pop_by_zip = df_pop[["zipcode","pop_0_5","pop_0_12"]].rename(columns={"zipcode":"zip_code"})

# ---------- 5. Income & employment ----------
inc_by_zip = df_inc.rename(columns={"ZIP code":"zip_code"})
emp_by_zip = df_emp.rename(columns={"zipcode":"zip_code"})

# ---------- 6. Merge ----------
df_zip = (
    cap_by_zip
    .merge(pop_by_zip, on="zip_code", how="left")
    .merge(inc_by_zip, on="zip_code", how="left")
    .merge(emp_by_zip, on="zip_code", how="left")
)

# ---------- 7. Cleaning ----------
nonzip_cols = [c for c in df_zip.columns if c != "zip_code"]
df_zip = df_zip.dropna(how="all", subset=nonzip_cols)
need_cols = ["existing_capacity_0_12","existing_capacity_0_5","pop_0_12","pop_0_5"]
df_zip = df_zip.dropna(subset=need_cols)

# ---------- 8. High-demand classification ----------
df_zip["high_demand"] = (df_zip["average income"] <= 60000) | (df_zip["employment rate"] >= 0.6)

# ---------- 9. Minimum required slots ----------
df_zip["min_required_0_12"] = np.where(
    df_zip["high_demand"],
    0.5 * df_zip["pop_0_12"],
    (1/3) * df_zip["pop_0_12"]
)
df_zip["min_required_0_5"] = (2/3) * df_zip["pop_0_5"]

# ---------- 10. Identify deserts ----------
df_zip["is_desert_0_12"] = df_zip["existing_capacity_0_12"] <= df_zip["min_required_0_12"]
df_zip["is_desert_0_5"]  = df_zip["existing_capacity_0_5"]  <= df_zip["min_required_0_5"]
df_zip["is_desert_any"]  = df_zip["is_desert_0_12"] | df_zip["is_desert_0_5"]
df_zip_opt = df_zip[df_zip["is_desert_any"]].copy()
print(f"Desert ZIPs: {df_zip_opt.shape[0]} / {df_zip.shape[0]}")

# ---------- 11. Parameters ----------
ZIPS = list(df_zip_opt["zip_code"].astype(int))
req_012 = df_zip_opt.set_index("zip_code")["min_required_0_12"].to_dict()
req_05  = df_zip_opt.set_index("zip_code")["min_required_0_5"].to_dict()

types       = ["small","medium","large"]
capacity    = {"small":100, "medium":200, "large":400}
capacity_05 = {"small":50, "medium":100, "large":200}
cost_build  = {"small":65000, "medium":95000, "large":115000}

# === Facility-level sets & dicts ===
F_tab = df_fac[df_fac["zip_code"].isin(ZIPS)][
    ["facility_id","zip_code","existing_capacity_0_12","existing_capacity_0_5"]
].copy()
F = list(F_tab["facility_id"])
zip_of_f = dict(zip(F_tab["facility_id"], F_tab["zip_code"]))
n12_f = dict(zip(F_tab["facility_id"], F_tab["existing_capacity_0_12"]))
n05_f = dict(zip(F_tab["facility_id"], F_tab["existing_capacity_0_5"]))

exist_zip_012 = F_tab.groupby("zip_code")["existing_capacity_0_12"].sum().to_dict()
exist_zip_005 = F_tab.groupby("zip_code")["existing_capacity_0_5"].sum().to_dict()
for z in ZIPS:
    exist_zip_012.setdefault(z, 0.0)
    exist_zip_005.setdefault(z, 0.0)

# ---------- 12. Build model ----------
m = Model("Q1_facility_level_v3")

# --- decision variables ---
# New builds (by ZIP)
y_build   = m.addVars(ZIPS, types, vtype=GRB.CONTINUOUS, lb=0.0, name="build")
z05_build = m.addVars(ZIPS, types, vtype=GRB.CONTINUOUS, lb=0.0, name="slots05_build")

# Facility-level expansion (piecewise as "added" capacity)
x1 = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="expand_add_0_100")   # up to +100% (added ≤ n_f)
x2 = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="expand_add_100_120") # extra +0–20% (added ≤ 0.2*n_f)
b  = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="trigger_over100")
z05_expand = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="slots05_from_expansion")

# --- capacity bounds for 0–5 allocation ---
for z in ZIPS:
    for t in types:
        m.addConstr(z05_build[z,t] <= capacity_05[t]*y_build[z,t], name=f"limit05_build_{z}_{t}")
for f in F:
    m.addConstr(z05_expand[f] <= x1[f] + x2[f], name=f"limit05_expand_{f}")

# --- piecewise expansion caps (added capacity): 0–100%, 100–120% ---
for f in F:
    nf = float(n12_f[f])
    u1 = 1.0 * nf
    u2 = 0.2 * nf

    # piecewise caps
    m.addConstr(x1[f] <= u1, name=f"x1cap_{f}")
    m.addConstr(x2[f] <= u2, name=f"x2cap_{f}")
    m.addConstr(x2[f] <= u2 * b[f], name=f"trigger_{f}")

    # 0–5 link
    m.addConstr(z05_expand[f] <= x1[f] + x2[f], name=f"limit05_expand_{f}")

    # NEW: 500-slot hard cap
    m.addConstr(x1[f] + x2[f] <= 500, name=f"cap500_total_{f}")

    
# --- coverage constraints ---
for z in ZIPS:
    expand12_z = quicksum(x1[f] + x2[f] for f in F if zip_of_f[f]==z)
    new12_z    = quicksum(capacity[t]*y_build[z,t] for t in types)
    m.addConstr(exist_zip_012[z] + expand12_z + new12_z >= req_012[z], name=f"cov012_{z}")

    new05_z = quicksum(z05_build[z,t] for t in types)
    exp05_z = quicksum(z05_expand[f] for f in F if zip_of_f[f]==z)
    m.addConstr(exist_zip_005[z] + new05_z + exp05_z >= req_05[z], name=f"cov05_{z}")

# ---------- 13. Objective ----------
# New build cost + equipment ($100/slot for both new & expansion 0–5)
build_cost = quicksum(cost_build[t]*y_build[z,t] for z in ZIPS for t in types)
equip_cost = (
    quicksum(100*z05_build[z,t] for z in ZIPS for t in types) +
    quicksum(100*z05_expand[f] for f in F)
)

# Expansion variable cost + one-time baseline fee if >100%
var_exp_cost = quicksum(
    (20000 + 200*n12_f[f])*(x1[f]/max(1.0,n12_f[f])) +
    (20000 + 200*n12_f[f])*(x2[f]/max(1.0,n12_f[f]))
    for f in F
)
baseline_fee = quicksum((20000 + 200*n12_f[f])*b[f] for f in F)

m.setObjective(build_cost + equip_cost + var_exp_cost + baseline_fee, GRB.MINIMIZE)

# ---------- 14. Solve LP then MIP ----------
print("🔹 Solving LP relaxation...")
m.optimize()
lp_obj = m.objVal
print(f"[LP] Objective = {lp_obj:,.2f}")

# Integerize (warm start from LP)
for var in y_build.values():   var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in z05_build.values(): var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in x1.values():        var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in x2.values():        var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in z05_expand.values():var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in b.values():         var.vType = GRB.BINARY;  var.Start = 1 if var.X>1e-6 else 0

m.update()
m.setParam("MIPFocus",1)
m.setParam("Heuristics",0.2)
m.setParam("Threads",0)
print("🔹 Solving MIP integer model...")
m.optimize()
mip_obj = m.objVal
print(f"[MIP] Objective = {mip_obj:,.2f}")

# ---------- 15. Export results ----------
rows=[]
for z in ZIPS:
    exp12 = sum((x1[f].X + x2[f].X) for f in F if zip_of_f[f]==z)
    new12 = sum(capacity[t]*y_build[z,t].X for t in types)
    new05 = sum(z05_build[z,t].X for t in types)
    exp05 = sum(z05_expand[f].X       for f in F if zip_of_f[f]==z)
    rows.append({
        "zip": int(z),
        "expand_0_12": exp12,
        "new_0_12":    new12,
        "new_0_5":     new05,
        "expand_0_5":  exp05,
        "cov012_LHS":  exist_zip_012[z] + exp12 + new12,
        "cov012_req":  req_012[z],
        "cov05_LHS":   exist_zip_005[z] + new05 + exp05,
        "cov05_req":   req_05[z],
    })

df_res = pd.DataFrame(rows).round(2)
out_path = "/Users/cuilinnan/Desktop/scenario1_solution_by_zip_facility_level.xlsx"
df_res.to_excel(out_path, index=False)
print(f"✅ Results saved to {out_path}")
print(df_res.head())


Desert ZIPs: 968 / 1066
🔹 Solving LP relaxation...
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 90712 rows, 63056 columns and 226295 nonzeros
Model fingerprint: 0xcbfa8201
Coefficient statistics:
  Matrix range     [6e-01, 4e+02]
  Objective range  [1e+02, 2e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+04]
Presolve removed 74973 rows and 19225 columns
Presolve time: 0.07s
Presolved: 15739 rows, 43831 columns, 83401 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 9.026e+03
 Factor NZ  : 4.216e+04 (roughly 10 MB of memory)
 Factor Ops : 4.814e+05 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Prima